In [1]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

ytt_api = YouTubeTranscriptApi()

In [3]:
video_id = 'syE8f_ow0iM'
print(ytt_api.list(video_id=video_id))

For this video (syE8f_ow0iM) transcripts are available in the following languages:

(MANUALLY CREATED)
None

(GENERATED)
 - en ("English (auto-generated)")[TRANSLATABLE]
 - fr-FR ("French (France) (auto-generated)")[TRANSLATABLE]
 - de-DE ("German (Germany) (auto-generated)")[TRANSLATABLE]
 - hi ("Hindi (auto-generated)")[TRANSLATABLE]
 - id ("Indonesian (auto-generated)")[TRANSLATABLE]
 - it ("Italian (auto-generated)")[TRANSLATABLE]
 - pt-BR ("Portuguese (Brazil) (auto-generated)")[TRANSLATABLE]
 - es-US ("Spanish (United States) (auto-generated)")[TRANSLATABLE]

(TRANSLATION LANGUAGES)
 - ar ("Arabic")
 - zh-Hant ("Chinese (Traditional)")
 - nl ("Dutch")
 - en ("English")
 - fr ("French")
 - de ("German")
 - hi ("Hindi")
 - id ("Indonesian")
 - it ("Italian")
 - ja ("Japanese")
 - ko ("Korean")
 - pt ("Portuguese")
 - ru ("Russian")
 - es ("Spanish")
 - th ("Thai")
 - uk ("Ukrainian")
 - vi ("Vietnamese")


In [4]:
try:
    transcript_list = ytt_api.fetch(video_id=video_id)
    print('Transcript list length: ', len(transcript_list), end='\n\n')
    print(transcript_list[0])
except TranscriptsDisabled:
    print('Transcript disabled for this video')

Transcript list length:  2404

FetchedTranscriptSnippet(text='What is up ladies and gentlemen? Welcome', start=0.24, duration=4.32)


In [5]:
transcript = ' '.join(chunk.text for chunk in transcript_list)
print('Total words in the transcript: ', len(transcript))

Total words in the transcript:  86691


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

In [7]:
chunks = splitter.create_documents(texts=[transcript])
print('Number of chunks: ', len(chunks), end='\n\n')
print('First chunk: ', chunks[0])


Number of chunks:  134

First chunk:  page_content='What is up ladies and gentlemen? Welcome back to Fudge Muppet. I'm Scott here with Michael and today we're talking all the law you need to know for Elder Scrolls 6. >> Yes. And I have a feeling this is going to be a series. Yeah. Series that people have been wanting for a long time. Dare I say the return of the Elder Scrolls 6 podcast. Maybe potentially probably. Now, we've been waiting for Elder Scrolls 6 for a very, very, very long time. In fact, since high school. We made this channel when we were wearing school uniforms and we would joke about how we'll have kids playing The Elder Scrolls 6. Well, I can tell you what, we need to wear glasses for this episode. >> We're not going to be able to read >> cringe glasses reveal, but we literally need to wear glasses. We have gray hairs on the'


___
Creating vector store:

In [77]:
from langchain_openai import OpenAIEmbeddings
import os

embedder = OpenAIEmbeddings(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",     # type: ignore
    model="local-bge-m3",
    check_embedding_ctx_length=False
)

# embedder = OpenAIEmbeddings(
#     model='nvidia/nemotron-3-embed-1b:free', 
#     base_url="https://openrouter.ai/api/v1",
#     check_embedding_ctx_length=False,
#     api_key=os.environ['OPENROUTER_API_KEY'] # type: ignore
# )

In [78]:
from dotenv import load_dotenv
load_dotenv()


from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedder,
    collection_name='chroma_collection',
)



KeyboardInterrupt: 

Indexing completed, Retrieval next

In [61]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={
        'k':10
    }
)

Augmentation

In [ ]:
from langchain_core.prompts import PromptTemplate

llm_prompt_template = PromptTemplate(
    template="""
You are a question-answering assistant.

Answer the question using ONLY the information contained in the CONTEXT below.

IMPORTANT:
- Do not use your own knowledge.
- Do not assume facts that are not present in the context.
- If the context does not contain enough information to answer the question, say "I don't know based on the provided context."
- The phrase "Great War" may refer to something specific in the provided context. Use the meaning given by the context, not your general knowledge.
- Do not worry about the source being asked from. For example, if the user asks about a "video" or a "blog", do not worry about the video itself. Assume the context given is from the video, and answer the remaining question accordingly. You will not have access to the source itself, just the transcript's relevant context from that source.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
""",
    input_variables=["context", "question"],
    validate_template=True
)

In [62]:
user_query = 'The Great War'

retrieved_docs = retriever.invoke(user_query)
context = ' '.join(chunk.page_content for chunk in retrieved_docs)

final_prompt = llm_prompt_template.invoke({'context': context, 'question':user_query})

Next, generation:

In [48]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

# llm = ChatOpenAI(
#     model='local qwen3.5:4b-8b',
#     base_url='http://127.0.0.1:8080/v1',
#     api_key='not needed' # type: ignore
# )

from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model='nvidia/nemotron-3.5-lightning:free'
)

Trying with a really good model using openrouter. Will later compare this output to a local qwen 3.5:4b model

In [49]:
result = llm.invoke(final_prompt)
print(result.content)

Based on the provided context, "The Great War" refers to a conflict in Elder Scrolls lore that begins toward the end of the Fourth Era (year 171). The transcript describes it as follows:

- The Aldmeri Dominion (referred to in the transcript as the "Old Mary Dominion") initiates the war by presenting the Empire with demands: cede southern Hammerfell, stop worshipping Talos, and pay a massive amount of gold. The Emperor refuses ("no"), triggering the war.
- The Dominion's initial strategy is to send forces into the southwest of Cyrodiil (around Anvil and Kabach) to draw Imperial troops away from supporting Hammerfell. They then launch aggressive attacks into southern Hammerfell, overwhelming the Legion and forcing a retreat.
- The withdrawing forces endure a prolonged, grueling period called the "March of Thirst," crossing the Alikir Desert and the central deserts of Hammerfell in a war of attrition. The Legion eventually regroups, but the Dominion has made significant territorial gains

In [63]:
retrieved_docs

[Document(id='740f0a16-d253-439d-8207-75a631db31a4', metadata={}, page_content="I think there's one of them that sounds like it's off the back of a it almost like on the back of Oblivion. That's the vibe. I can see the font, you know, the cover of the game. uh where it talks about the next coming war. >> Yeah, I can um I can find it here, but it's in the in the in the Great War book. So, it's like right at the end. There's literally a section called epilogue Hammerfell fights on alone, and it goes, you know, well, maybe we should should we explain the Great War first, a little bit of the context first. All right. So, we'll start us off with >> So, if it's Let's just assume it's Hammerfell. This is the Elder Scrolls 6. It's going to be Hammerfell podcast. We we can touch on some of the other things, but we're just going to >> small plug real quick. Upcoming episodes"),
 Document(id='e0970a6a-aef0-418d-aee6-926d7f7af835', metadata={}, page_content="but anyway so we should all be pretty f

In [66]:
# 
local_llm = ChatOpenAI(
    model='local qwen3.5:4b-8b',
    base_url='http://127.0.0.1:8080/v1',
    api_key='not needed' # type: ignore
)

In [67]:
local_results = local_llm.invoke(final_prompt)
print(local_results.content)

Based on the provided context, here is the information regarding the Great War:

*   **Timeframe:** The Great War started towards the end of the Fourth Era (171).
*   **Causes:** It began after the Old Mary Dominion showed up at Sidril and demanded the Emperor give them the south of Hammerfell, stop the worship of Talos, and provide a lot of money. The Emperor refused, saying "no."
*   **The Old Mary Dominion's Goal:** Their original goal was to take Hammerfell. They sent a force into the southwest of Sidril (around Anvil, Kabach, and similar areas) to draw Imperial forces away from supporting Hammerfell. They attacked aggressively into the south of Hammerfell, which caused the Legion to become outnumbered ("pawned") and forced them to retreat across the big Alakir desert.
*   **The "March of Thirst":** During the conflict, there was a long and slow war of attrition in the central deserts of Hammerfell.
*   **The Outcome/Conclusion:** The Old Mary Dominion's attempt to take Hammerfell 

### Clearly, keeping everything else the exact same and only trying slightly different models, it is clear that the cloud model is much better

This was tried with reasoning off on the local qwen3.5, but with reasoning on, it was an even bigger problem. The model just kept stuck in a loop of "Wait, have I checked this", and "Yes, but then have I done this again". So decided to try with reasoning off

___
Building a chain for this. Will use openrouter models for this, although anyone with a better GPU can use a local LLM and get good performance. My preference will always be local models, but since I cannot run good models on my RTX3050 with 4GB VRAM, I will use Openrouter here  

Promp -> LLM -> Output: Very simple chain, just a RunnableSequence <br><br>

Will need another parallel chain for the prompt though, since you have two input fields for the prompt, the user question and the retrieved chunks

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

from typing import List
from langchain_core.documents import Document, str

In [71]:
def join_retrieved_docs(retrieved_docs: List[Document]) -> str:
    return ' '.join(chunk.page_content for chunk in retrieved_docs)

In [72]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(join_retrieved_docs),
    'question': RunnablePassthrough()
})

In [74]:
str_parser = StrOutputParser()

sequence_chain = parallel_chain | llm_prompt_template | llm | str_parser

In [75]:
final_chain = parallel_chain | sequence_chain # type: ignore

In [ ]:
user_query = input('Ask something')

final_chain.invoke(user_query)